# Simplex Solver -- Demo Notebook
### Originally developed as part of the 4th Semester course "Optimization"

This notebook demonstrates the general-purpose `simplex_solver` package
(see the `simplex_solver/` directory), which replaces the single-file
teaching implementation that used to live directly in this notebook.

Compared to the original teaching version, `simplex_solver` supports:
- `<=`, `>=`, and `=` constraints (via a two-phase method with artificial variables)
- both **maximize** and **minimize** objectives
- free and lower/upper-bounded variables
- negative right-hand sides
- explicit `OPTIMAL` / `INFEASIBLE` / `UNBOUNDED` status reporting
- an anti-cycling pivoting rule (Bland's rule fallback) and an incremental
  (product-form-of-the-inverse) basis update instead of recomputing a full
  matrix inverse every iteration

The original explicit-matrix-inversion, `<=`-only teaching algorithm is
preserved verbatim in `simplex_solver/legacy_teaching.py` for comparison --
see the last section of this notebook.

In [ ]:
from simplex_solver import solve, LPProblem, Constraint, Status


## Example 1: canonical `<=` problem

The same problem as `vl.txt`: maximize `3x + 2y` subject to three `<=`
constraints. This is exactly the kind of problem the original teaching
implementation could already solve -- included here to confirm the new
solver reproduces the same result (`7.25`).

In [ ]:
problem = LPProblem(
    c=[3, 2],
    sense="maximize",
    constraints=[
        Constraint(coeffs=[4, 2], op="<=", rhs=9),
        Constraint(coeffs=[10, 20], op="<=", rhs=51),
        Constraint(coeffs=[4, 3], op="<=", rhs=10),
    ],
)
result = solve(problem)
print(f"Status: {result.status}")
print(f"x: {result.x}")
print(f"Objective value: {result.objective_value}")


Status: Status.OPTIMAL
x: [1.75, 1.0]
Objective value: 7.25


## Example 2: mixed `<=`, `=`, and `>=` constraints

This is new: the original implementation only ever parsed `<=` rows. Here
a `>=` and an `=` constraint are handled via the two-phase method (Phase I
finds an initial feasible basis using artificial variables before Phase II
optimizes the real objective).

In [ ]:
problem = LPProblem(
    c=[1, 1],
    sense="maximize",
    constraints=[
        Constraint(coeffs=[1, 1], op="<=", rhs=10),
        Constraint(coeffs=[1, -1], op="=", rhs=2),
        Constraint(coeffs=[1, 2], op=">=", rhs=3),
    ],
)
result = solve(problem)
print(f"Status: {result.status}")
print(f"x: {result.x}")
print(f"Objective value: {result.objective_value}")


Status: Status.OPTIMAL
x: [6.0, 4.0]
Objective value: 10.0


## Example 3: minimization

The original notebook's docstring said: *"objective function is always
maximized; if you want to minimize instead, simply turn your minimization
problem into a maximization problem [yourself]"*. `simplex_solver` handles
`sense="minimize"` natively instead.

In [ ]:
problem = LPProblem(
    c=[1, 1],
    sense="minimize",
    constraints=[
        Constraint(coeffs=[1, 1], op=">=", rhs=4),
        Constraint(coeffs=[1, -1], op="=", rhs=0),
    ],
)
result = solve(problem)
print(f"Status: {result.status}")
print(f"x: {result.x}")
print(f"Objective value: {result.objective_value}")


Status: Status.OPTIMAL
x: [2.0, 2.0]
Objective value: 4.0


## Example 4: infeasible and unbounded detection

The original implementation had no way to represent an infeasible problem
at all (it assumed a `<=`/`b >= 0` feasible starting basis always exists),
and reported unboundedness by raising a bare `ValueError`. `simplex_solver`
reports both as an explicit `Status` on the result instead.

In [ ]:
infeasible_problem = LPProblem(
    c=[1, 1],
    sense="maximize",
    constraints=[
        Constraint(coeffs=[1, 1], op="<=", rhs=1),
        Constraint(coeffs=[1, 1], op=">=", rhs=5),
    ],
)
print("Infeasible problem status:", solve(infeasible_problem).status)

unbounded_problem = LPProblem(
    c=[1],
    sense="maximize",
    constraints=[Constraint(coeffs=[-1], op="<=", rhs=0)],
)
print("Unbounded problem status:", solve(unbounded_problem).status)


Infeasible problem status: Status.INFEASIBLE
Unbounded problem status: Status.UNBOUNDED


## Legacy vs. new solver, side by side

`simplex_solver/legacy_teaching.py` is a verbatim, unmodified port of this
notebook's original algorithm cells (explicit `A_B_inv` recomputation every
iteration, `<=`-only, maximize-only). It's kept around specifically so it
can be compared against the general-purpose solver on the same input.

In [ ]:
from simplex_solver.legacy_teaching import simplex as legacy_simplex

_, _, legacy_objective, optimal, basis_indices, non_basis_indices = legacy_simplex("vl.txt")
print(f"legacy_teaching.simplex(): optimal={optimal}, objective={legacy_objective}")

problem = LPProblem(
    c=[3, 2],
    sense="maximize",
    constraints=[
        Constraint(coeffs=[4, 2], op="<=", rhs=9),
        Constraint(coeffs=[10, 20], op="<=", rhs=51),
        Constraint(coeffs=[4, 3], op="<=", rhs=10),
    ],
)
result = solve(problem)
print(f"simplex_solver.solve():     status={result.status}, objective={result.objective_value}")


A_B shape: (3, 3)
A_N shape: (3, 2)
m = 3 , n = 2
legacy_teaching.simplex(): optimal=True, objective=7.25
simplex_solver.solve():     status=Status.OPTIMAL, objective=7.25
